In [1]:
import os

# 1. Cấp quyền cho file kaggle.json bạn vừa upload
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# 2. Tải Dataset SisFall từ Kaggle
print("Đang tải bộ dữ liệu SisFall...")
!kaggle datasets download -d thevman/sisfall-dataset

# 3. Tạo thư mục và giải nén toàn bộ file txt vào một chỗ
print("Đang giải nén dữ liệu...")
!mkdir -p /content/SisFall_Data
!unzip -q sisfall-dataset.zip -d /content/SisFall_Data/

# Lệnh này giúp lôi tất cả các file .csv từ các thư mục con ra ngoài cùng cho dễ đọc
!find /content/SisFall_Data/ -name "*.csv" -exec mv {} /content/SisFall_Data/ \;

print("Xong! Dữ liệu đã sẵn sàng.")

cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
Đang tải bộ dữ liệu SisFall...
Dataset URL: https://www.kaggle.com/datasets/thevman/sisfall-dataset
License(s): Attribution 4.0 International (CC BY 4.0)
100% 553M/553M [00:27<00:00, 20.8MB/s]

Đang giải nén dữ liệu...
Xong! Dữ liệu đã sẵn sàng.


In [2]:
import os
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

# ==========================================
# 1. TÌM TẤT CẢ FILE CẢM BIẾN TRONG COLAB
# ==========================================
file_paths = []
for root, dirs, files in os.walk('/content'):
    for file in files:
        if file.lower().endswith(('.txt', '.csv')) and (file.startswith('F') or file.startswith('D')):
            file_paths.append(os.path.join(root, file))

print(f"Tổng số file tìm thấy: {len(file_paths)}")

# ==========================================
# 2. CẤU HÌNH DOWNSAMPLE VÀ WINDOW
# ==========================================
TARGET_HZ = 50
ORIGINAL_HZ = 200
DOWNSAMPLE_RATE = ORIGINAL_HZ // TARGET_HZ  # = 4
WINDOW_SIZE = 128
STEP_SIZE = 64

X_list = []
y_list = []

for file_path in file_paths:
    filename = os.path.basename(file_path)
    is_fall_file = filename.startswith('F')

    try:
        # Đọc file thô linh hoạt mọi loại dấu phân cách, bỏ qua các dòng lỗi
        df = pd.read_csv(file_path, sep=r'[,\s;]+', engine='python', on_bad_lines='skip')

        # Chỉ lấy 6 cột đầu tiên (để né cột thứ 7 toàn rác do dấu chấm phẩy ở cuối dòng gây ra)
        # Ép kiểu sang float32 và loại bỏ các hàng chứa giá trị rỗng (NaN)
        df = df.iloc[:, 0:6].apply(pd.to_numeric, errors='coerce').dropna()
        data = df.values.astype(np.float32)

        # Bỏ qua các file quá ngắn (dưới 1 giây gốc)
        if len(data) < ORIGINAL_HZ:
            continue

        # Downsample từ 200Hz xuống 50Hz (Lấy 1 mẫu cho mỗi 4 mẫu)
        data_50hz = data[::DOWNSAMPLE_RATE, :]
        num_samples = len(data_50hz)

        # Bỏ qua các file sau khi downsample bị ngắn hơn 1 window (128)
        if num_samples < WINDOW_SIZE:
            continue

        # ==========================================
        # 3. THUẬT TOÁN CẮT CỬA SỔ (SLIDING WINDOW & PEAK DETECTION)
        # ==========================================
        if is_fall_file:
            # --- FILE NGÃ: TÌM ĐỈNH GIA TỐC (PEAK DETECTION) ---
            # Tính độ lớn gia tốc tổng hợp
            accel_magnitude = np.sqrt(data_50hz[:,0]**2 + data_50hz[:,1]**2 + data_50hz[:,2]**2)

            # Cắt bỏ 50 mẫu (1 giây) ở hai đầu để tránh nhiễu do cầm/đặt thiết bị
            MARGIN = 50
            if num_samples > 2 * MARGIN:
                search_area = accel_magnitude[MARGIN:-MARGIN]
                peak_idx = np.argmax(search_area) + MARGIN
            else:
                peak_idx = np.argmax(accel_magnitude)

            # Tính toán tọa độ khung cửa sổ (Window) bao quanh đỉnh ngã
            start = peak_idx - (WINDOW_SIZE // 2)
            end = peak_idx + (WINDOW_SIZE // 2)

            # THUẬT TOÁN ÉP KHUNG (Chống lọt viền làm mất dữ liệu)
            if start < 0:
                start = 0
                end = WINDOW_SIZE
            elif end > num_samples:
                end = num_samples
                start = end - WINDOW_SIZE

            # Xác nhận lần cuối độ dài đúng 128 mẫu thì mới đưa vào danh sách
            if end - start == WINDOW_SIZE:
                X_list.append(data_50hz[start:end, :])
                y_list.append(1) # Label 1: Ngã

        else:
            # --- FILE KHÔNG NGÃ (ADL): SLIDING WINDOW BÌNH THƯỜNG ---
            for start in range(0, num_samples - WINDOW_SIZE + 1, STEP_SIZE):
                end = start + WINDOW_SIZE
                X_list.append(data_50hz[start:end, :])
                y_list.append(0) # Label 0: Không ngã

    except Exception:
        continue

X = np.array(X_list)
y = np.array(y_list)

# ==========================================
# 4. IN BÁO CÁO KẾT QUẢ TIỀN XỬ LÝ
# ==========================================
print("\n--- KẾT QUẢ TIỀN XỬ LÝ THÀNH CÔNG ---")
print(f"Kích thước ma trận X (Input Model): {X.shape}")
if len(y) > 0:
    print(f"Số mẫu Không ngã (ADL - Label 0): {np.sum(y == 0)}")
    print(f"Số mẫu Ngã (Fall - Label 1): {np.sum(y == 1)}")
else:
    print("🚨 Lỗi: Mảng dữ liệu vẫn rỗng! Hãy kiểm tra lại file của bạn.")

Tổng số file tìm thấy: 4275

--- KẾT QUẢ TIỀN XỬ LÝ THÀNH CÔNG ---
Kích thước ma trận X (Input Model): (15673, 128, 6)
Số mẫu Không ngã (ADL - Label 0): 13882
Số mẫu Ngã (Fall - Label 1): 1791


In [3]:
import numpy as np
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping # Thêm cơ chế tự dừng
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.layers import LeakyReLU

# 1. CHIA TRAIN/TEST TRƯỚC! (Tuyệt đối không rò rỉ dữ liệu)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 2. CHUẨN HÓA Z-SCORE SAU KHI CHIA (Chỉ fit trên Train)
mean_train = np.mean(X_train, axis=(0, 1), keepdims=True)
std_train = np.std(X_train, axis=(0, 1), keepdims=True)

X_train_scaled = (X_train - mean_train) / (std_train + 1e-8)
# Dùng mean_train và std_train để biến đổi X_test
X_test_scaled = (X_test - mean_train) / (std_train + 1e-8)

# 3. Class Weight (Tốt)
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = dict(enumerate(class_weights))

# 4. Xây dựng Model - Tối ưu cho ESP32/STM32 (Binary Classification)
model = models.Sequential([
    layers.Conv1D(32, kernel_size=3, padding='same', input_shape=(128, 6)),
    LeakyReLU(alpha=0.01),
    layers.MaxPooling1D(pool_size=2),

    layers.Conv1D(64, kernel_size=3, padding='same'),
    LeakyReLU(alpha=0.01),
    layers.MaxPooling1D(pool_size=2),

    layers.Flatten(),
    # Giảm bớt số lượng Dense layer để bớt ăn RAM của MCU
    layers.Dense(64),
    LeakyReLU(alpha=0.01),
    layers.Dropout(0.4),
    # CHỈ DÙNG 1 NƠ-RON VÀ HÀM SIGMOID
    layers.Dense(1, activation='sigmoid')
])

# Sửa lại hàm Loss cho phù hợp với Sigmoid
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# 5. Cài đặt giám thị - EarlyStopping
# Nếu validation loss không giảm sau 15 epochs, tự động dừng lại!
early_stop = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

# 6. Huấn luyện
print("\nBắt đầu huấn luyện...")
history = model.fit(
    X_train_scaled, y_train,
    epochs=200, # Cứ để 200, EarlyStopping sẽ tự ngắt sớm
    batch_size=32,
    validation_split=0.15,
    class_weight=class_weight_dict,
    callbacks=[early_stop], # Kích hoạt EarlyStopping
    verbose=1
)

# 7. Đánh giá
print("\n--- BÁO CÁO ĐÁNH GIÁ MÔ HÌNH ---")
# Hàm Sigmoid trả về xác suất, ta làm tròn > 0.5 là 1 (Ngã), <= 0.5 là 0
y_pred_prob = model.predict(X_test_scaled)
y_pred = (y_pred_prob > 0.5).astype(int).flatten()

print(classification_report(y_test, y_pred, target_names=['ADL (0)', 'Fall (1)']))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))



Bắt đầu huấn luyện...
Epoch 1/200
334/334 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - accuracy: 0.8550 - loss: 0.3776 - val_accuracy: 0.9155 - val_loss: 0.2527
Epoch 2/200
334/334 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9247 - loss: 0.2459 - val_accuracy: 0.9234 - val_loss: 0.2304
Epoch 3/200
334/334 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9439 - loss: 0.2091 - val_accuracy: 0.9532 - val_loss: 0.1618
Epoch 4/200
334/334 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9505 - loss: 0.1788 - val_accuracy: 0.9410 - val_loss: 0.1494
Epoch 5/200
334/334 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9525 - loss: 0.1601 - val_accuracy: 0.9484 - val_loss: 0.1652
Epoch 6/200
334/334 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.9558 - loss: 0.1494 - val_accuracy: 0.9617 - val_loss: 0.1322
Epoch 7/200
334/334 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9560 - loss: 0.1439 - val_accuracy: 0.9601 - val_loss: 0.1426
Epoch 8/200
334/334 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9620 -

In [4]:
import tensorflow as tf

print("1. Đang chuyển đổi sang TensorFlow Lite...")
# Khởi tạo bộ chuyển đổi từ model Keras của bạn
converter = tf.lite.TFLiteConverter.from_keras_model(model)

# 2. BẬT LƯỢNG TỬ HÓA (QUANTIZATION) ĐỂ TỐI ƯU CHO MCU
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# Thực hiện chuyển đổi
tflite_model = converter.convert()

# Lưu thành file .tflite để backup
with open('fall_model_quantized.tflite', 'wb') as f:
    f.write(tflite_model)
print(f"Kích thước file TFLite: {len(tflite_model)} bytes (~{len(tflite_model)/1024:.2f} KB)")

# ==========================================
# 3. CHUYỂN ĐỔI THÀNH FILE C++ HEADER (.h)
# ==========================================
print("\nĐang tạo file model.h cho ESP32/STM32...")

def hex_to_c_array(hex_data, var_name):
    c_str = f"#ifndef MODEL_H_\n#define MODEL_H_\n\n"
    c_str += f"// Kích thước mô hình: {len(hex_data)} bytes\n"
    c_str += f"const unsigned char {var_name}[] = {{\n"
    hex_array = [f"0x{b:02x}" for b in hex_data]

    # Chia mỗi dòng 12 byte cho đẹp code
    for i in range(0, len(hex_array), 12):
        c_str += "  " + ", ".join(hex_array[i:i+12]) + ",\n"

    c_str += "};\n\n"
    c_str += f"const unsigned int {var_name}_len = {len(hex_data)};\n\n"
    c_str += "#endif // MODEL_H_\n"
    return c_str

# Tạo file C header
c_code = hex_to_c_array(tflite_model, "fall_detect_model")

with open('model.h', 'w') as f:
    f.write(c_code)

print("✅ Đã xuất thành công file model.h!")
print("Bạn có thể tải file model.h ở cột thư mục bên trái của Google Colab.")

1. Đang chuyển đổi sang TensorFlow Lite...
Saved artifact at '/tmp/tmp7t3obcu4'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 128, 6), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  132519532321232: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132519532322192: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132519532324688: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132519532323344: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132519532324112: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132519532325456: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132519532325840: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132519532326416: TensorSpec(shape=(), dtype=tf.resource, name=None)
Kích thước file TFLite: 148504 bytes (~145.02 KB)

Đang tạo file model.h cho ESP32/STM32...
✅ Đã xuất thành công file 

### 1. Model Splitting & Architecture Setup
Chúng ta sẽ chia mô hình hiện tại (giả định là `model`) thành 2 phần: Trích xuất đặc trưng (cố định) và Bộ phân loại (huấn luyện online).

In [7]:
import tensorflow as tf
import numpy as np

if 'model' not in locals():
    print("🚨 LỖI: Chưa có mô hình. Hãy chạy lại Cell 3 trước nhé!")
else:
    # ==================================================
    # YÊU CẦU 1: CẮT MÔ HÌNH (TƯƠNG THÍCH KÈM KERAS 3 TRÊN COLAB)
    # ==================================================

    # 1. XÂY DỰNG FEATURE EXTRACTOR (ĐÓNG BĂNG - NẠP VÀO FLASH ESP32)
    feature_extractor = tf.keras.Sequential(name="Frozen_Feature_Extractor")

    # Định nghĩa Input rõ ràng để Keras 3 không bị lỗi
    feature_extractor.add(tf.keras.layers.Input(shape=(128, 6)))

    # Copy tất cả các lớp từ model gốc, NHƯNG BỎ QUA 2 LỚP CUỐI CÙNG (Dropout và Output)
    # Nghĩa là ta lấy từ đầu cho đến hết lớp LeakyReLU sau Dense(64)
    for layer in model.layers[:-2]:
        layer.trainable = False # Đóng băng trọng số
        feature_extractor.add(layer)

    # 2. XÂY DỰNG ONLINE CLASSIFIER (HỌC LIÊN TỤC TRÊN SRAM ESP32)
    # Lấy kích thước output của Feature Extractor làm input cho Classifier
    latent_dim = feature_extractor.output_shape[-1]

    online_classifier = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(latent_dim,)),
        tf.keras.layers.Dense(1, activation='sigmoid', name="Online_Classifier_Trainable")
    ])

    # 3. Chuyển giao "kiến thức": Copy trọng số của lớp cuối cùng từ model cũ sang
    original_weights = model.layers[-1].get_weights()
    online_classifier.layers[0].set_weights(original_weights)

    # ==================================================
    # IN BÁO CÁO PHẪU THUẬT MÔ HÌNH
    # ==================================================
    print("\n✅ --- PHẪU THUẬT MÔ HÌNH THÀNH CÔNG --- ✅")
    print(f"Kích thước Latent Vector (Dữ liệu nén): {latent_dim} đặc trưng")

    print("\n[PHẦN 1] Tính toán NẶNG - Lưu vào bộ nhớ tĩnh (Flash):")
    feature_extractor.summary()

    print("\n[PHẦN 2] Tính toán NHẸ - Học trực tiếp trên RAM (SRAM):")
    online_classifier.summary()


✅ --- PHẪU THUẬT MÔ HÌNH THÀNH CÔNG --- ✅
Kích thước Latent Vector (Dữ liệu nén): 64 đặc trưng

[PHẦN 1] Tính toán NẶNG - Lưu vào bộ nhớ tĩnh (Flash):


Model: "Frozen_Feature_Extractor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 128, 32)        │           608 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu (LeakyReLU)         │ (None, 128, 32)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 64, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 64, 64)         │         6,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_1 (LeakyReLU)       │ (None, 64, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 32, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │       131,136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_2 (LeakyReLU)       │ (None, 64)             │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 137,952 (538.88 KB)

 Trainable params: 0 (0.00 B)

...

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ Online_Classifier_Trainable     │ (None, 1)              │            65 │
│ (Dense)                         │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 65 (260.00 B)

 Trainable params: 65 (260.00 B)

 Non-trainable params: 0 (0.00 B)

### 2. Latent Replay Buffer
Lớp này dùng để lưu trữ các vector đặc trưng (latent vectors) thay vì dữ liệu raw để tiết kiệm RAM.

In [10]:
class LatentReplayBuffer:
    def __init__(self, max_size=1000):
        self.max_size = max_size
        self.buffer_x = []  # Lưu latent vectors (64,)
        self.buffer_y = []  # Lưu nhãn (0 hoặc 1)

    def add(self, latent_vector, label):
        # Nếu buffer đầy, xóa mẫu cũ nhất (FIFO)
        if len(self.buffer_x) >= self.max_size:
            self.buffer_x.pop(0)
            self.buffer_y.pop(0)

        # Ép kiểu về numpy để lưu trữ nhẹ hơn tensor
        self.buffer_x.append(np.array(latent_vector).flatten())
        self.buffer_y.append(label)

    def sample(self, batch_size):
        if len(self.buffer_x) < batch_size:
            batch_size = len(self.buffer_x)

        # Lấy chỉ số ngẫu nhiên
        indices = random.sample(range(len(self.buffer_x)), batch_size)

        batch_x = np.array([self.buffer_x[i] for i in indices])
        batch_y = np.array([self.buffer_y[i] for i in indices]).reshape(-1, 1)

        return tf.convert_to_tensor(batch_x, dtype=tf.float32), \
               tf.convert_to_tensor(batch_y, dtype=tf.float32)

# Khởi tạo buffer
replay_buffer = LatentReplayBuffer(max_size=1000)

### 3. Online Training Step (Trigger SGD)
Hàm này mô phỏng việc thiết bị nhận phản hồi từ người dùng và tự cập nhật.

In [11]:
# Cấu hình bộ tối ưu hóa nhỏ nhẹ cho thiết bị nhúng
optimizer_online = tf.keras.optimizers.SGD(learning_rate=0.001)
loss_fn = tf.keras.losses.BinaryCrossentropy()

def on_device_training_step(new_X_raw, user_corrected_label, batch_size=16):
    """
    Mô phỏng 1 bước học lại khi có dữ liệu mới từ sensor và feedback người dùng
    """
    # 1. Chuyển đổi dữ liệu thô sang Latent Vector (Feature Extraction)
    # new_X_raw shape: (1, 128, 6)
    latent_feat = feature_extractor(new_X_raw, training=False)

    # 2. Lưu vào Buffer để chống quên kiến thức cũ
    replay_buffer.add(latent_feat, user_corrected_label)

    # 3. Lấy mẫu từ Buffer để huấn luyện (Latent Replay)
    if len(replay_buffer.buffer_x) >= batch_size:
        x_batch, y_batch = replay_buffer.sample(batch_size)

        with tf.GradientTape() as tape:
            predictions = online_classifier(x_batch, training=True)
            loss = loss_fn(y_batch, predictions)

        # 4. Cập nhật trọng số chỉ cho lớp Dense cuối
        grads = tape.gradient(loss, online_classifier.trainable_variables)
        optimizer_online.apply_gradients(zip(grads, online_classifier.trainable_variables))

        return loss.numpy()
    return None

# --- MÔ PHỎNG THỬ NGHIỆM ---
print("Mô phỏng 1 bước học: ")
# Lấy 1 mẫu ngẫu nhiên từ tập test để giả lập dữ liệu sensor
test_sample = X_test_scaled[0:1]
test_label = y_test[0]

loss_val = on_device_training_step(test_sample, test_label)
if loss_val:
    print(f"Cập nhật thành công! Loss hiện tại: {loss_val:.4f}")
else:
    print("Buffer chưa đủ dữ liệu để huấn luyện.")

Mô phỏng 1 bước học: 
Buffer chưa đủ dữ liệu để huấn luyện.


### 4. Kiểm chứng: Mô phỏng quá trình Học liên tục (Continual Learning Simulation)
Cell này sẽ chạy thử nghiệm việc học online qua 50 mẫu dữ liệu mới để kiểm tra xem hệ thống có hoạt động ổn định không.

In [12]:
import time

print("--- BẮT ĐẦU MÔ PHỎNG HỌC LIÊN TỤC ---")

# Giả lập việc nhận 50 mẫu dữ liệu mới từ sensor theo thời gian thực
for i in range(50):
    # 1. Lấy ngẫu nhiên 1 mẫu từ tập Test để giả làm dữ liệu sensor
    idx = np.random.randint(0, len(X_test_scaled))
    sample_raw = X_test_scaled[idx : idx + 1] # Shape (1, 128, 6)
    true_label = y_test[idx]

    # 2. Gọi hàm học trên thiết bị (Trigger SGD)
    # Chúng ta giả định người dùng cung cấp nhãn đúng (label)
    loss = on_device_training_step(sample_raw, true_label, batch_size=16)

    if (i + 1) % 10 == 0:
        buffer_fill = len(replay_buffer.buffer_x)
        loss_str = f"{loss:.4f}" if loss is not None else "Đang làm đầy buffer..."
        print(f"Mẫu thứ {i+1:02d} | Buffer: {buffer_fill}/1000 | Loss: {loss_str}")
        time.sleep(0.1) # Giả lập độ trễ xử lý

print("\n✅ Hoàn tất mô phỏng. Hệ thống đã sẵn sàng tích hợp vào phần cứng!")

--- BẮT ĐẦU MÔ PHỎNG HỌC LIÊN TỤC ---
Mẫu thứ 10 | Buffer: 11/1000 | Loss: Đang làm đầy buffer...
Mẫu thứ 20 | Buffer: 21/1000 | Loss: 0.0132
Mẫu thứ 30 | Buffer: 31/1000 | Loss: 0.0205
Mẫu thứ 40 | Buffer: 41/1000 | Loss: 0.0235
Mẫu thứ 50 | Buffer: 51/1000 | Loss: 0.1053

✅ Hoàn tất mô phỏng. Hệ thống đã sẵn sàng tích hợp vào phần cứng!
